# New version (September 2026) for processing standard MOUSE files using the MoDaCor server

This notebook processes MOUSE X-ray scattering measurements with the MoDaCor runtime server. We use this server workflow when many similar files should reuse cached runtime state. In the example here we have one sample and one background, both consisting of 38 measurements across a number of instrument configurations. These 38 measurements are combined using the standard MOUSE preprocessing scripts into a reduced number of stacked files: typically 10 when only copper-source measurements have been carried out, and a few more when molybdenum measurements are included. Copper configurations have odd hundreds digits in the configuration number, while molybdenum configurations have even hundreds digits. The wavelength and source can also be derived from the extensive instrument metadata in the HDF5 files.

For this example, the sample files start at batch number 2. Stacked filenames typically contain a six-digit year-month-day code for the measurement series, followed by a batch number and a three-digit configuration number. These numbers are repeated in the internal metadata. Files ending in `.nxs` are HDF5 files.

For MoDaCor, the stacked files are further preprocessed using a separate script to produce MoDaCor-ready files. This is needed because MoDaCor has stricter casting and broadcasting rules than the previous DAWN implementation, so some arrays need to be extended before mathematical operations can be applied. These files end in `_stacked_modacor.nxs`.

For each configuration number, a correction-pipeline instance using the same universal pipeline YAML will be created in a separate runtime session on the server. Each session receives the corresponding stacked background and sample files. The notebook is intended to process a series of samples. Background and optional dispersed-background files are specified inside each measurement file, but currently still point to the non-MoDaCor stacked files. Until that upstream reference changes, `_modacor` must be added to a referenced background file's stem when necessary.

The target is the full correction sequence described in our 2017 modular data-correction pipeline paper ([open access](https://journals.iucr.org/j/issues/2017/06/00/vg5075/index.html)). The draft pipeline is `pipelines/MOUSE_solids.yaml`. It now masks measured signals outside the inclusive range 0 through 1e6 and flatfield values outside 0.96 through 1.04, dilates the assembled mask by one pixel, applies the embedded flatfield matrix, corrects detector efficiency from the embedded sensor and wavelength metadata, applies the polarization correction for an unpolarized source, and corrects sample self-absorption for plate-like samples from their measured transmission. Available detector, transmission, geometry, self-absorption, and thickness uncertainties are propagated under distinct source names. It still needs live plot outputs following the I22 `solids_operando` examples.

The notebook discovers and validates the copied example files, renders the pipeline graph, performs a local smoke test, and processes the selected batches through configuration-specific runtime sessions. Run the cells from top to bottom; for normal use, edit only **User configuration**.

Fresh environment setup from a terminal:

```bash
git clone https://github.com/BAMresearch/MoDaCor.git
cd MoDaCor
uv venv --python 3.12 .venv
source .venv/bin/activate
uv pip install -e ".[server,attenuation,plotting]" requests matplotlib ipykernel
python -m ipykernel install --user --name modacor-mouse --display-name "Python (MoDaCor MOUSE)"
cd -
```

After installation, select the `Python (MoDaCor MOUSE)` kernel for this notebook.


## Uncertainty metadata still needed

The pipeline propagates every uncertainty currently available in the converted files. The following correction inputs do not yet have usable uncertainty datasets and should be added upstream when estimates become available:

- Total detector count time
- Incident flux
- Dark-current correction
- Flatfield correction matrix
- Detector-position coordinates and rotations
- Detector sensor thickness

Do not use detector readout time as the uncertainty of total count time; it is an acquisition timing parameter, not an uncertainty estimate. Existing detector Poisson and SEM terms, sample/background transmission SEM, wavelength error, self-absorption transmission sensitivity, and sample-thickness SEM are already propagated.


## Preparing stacked MOUSE files

For new data, install the MOUSE conversion utility and run it in the directory containing the stacked files:

```bash
uv pip install git+https://github.com/BAMresearch/MOUSEDataPipeline.git@bluesky_mods
mouse-stacked-to-modacor MOUSE_*_stacked.nxs
```

The converter creates files ending in `_stacked_modacor.nxs`. No extra I22-style calibration or normalization preprocessing is needed: MOUSE geometry, masks, corrections, and measurement metadata are embedded in each file.


## User configuration

Start Jupyter from the examples repository root or this instrument directory. The packaged data and pipelines are then discovered automatically. Point `DATA_ROOT` at another measurement directory only when processing data outside this example.


In [ ]:
from pathlib import Path

def locate_example_dir(relative_path):
    start = Path.cwd().resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / relative_path):
            if (candidate / "data-manifest.json").is_file() and (candidate / "pipelines").is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not locate BAM/MOUSE. Start Jupyter from the examples repository root "
        "or from the BAM/MOUSE instrument directory."
    )


PROJECT_DIR = locate_example_dir(Path("BAM") / "MOUSE")
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "MOUSE_solids.yaml"
DATA_ROOT = PROJECT_DIR / "data"
SAMPLE_GLOB = "MOUSE_*_stacked_modacor.nxs"
SAMPLE_BATCH_START = 2
SAMPLE_BATCH_END = 2
BACKGROUND_REFERENCE_PATH = "/entry1/processing_required_metadata/background_file"
DISPERSANT_REFERENCE_PATH = "/entry1/processing_required_metadata/dispersed_background_file"
DISPLACED_DISPERSANT_FACTOR_PATH = "/entry1/sample/matrixfraction"
EMBEDDED_MASK_PATH = "/entry1/instrument/mask/Mask"
OUTPUT_DIR = PROJECT_DIR / "work" / "output"
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
TRACE_ENABLED = True
OUTPUT_DATA_PATHS = ["/sample/signal", "/sample/Q"]


## Environment and data discovery


In [ ]:
import sys

import h5py
import modacor

for label, path in {"project": PROJECT_DIR, "pipeline": PIPELINE_PATH, "data": DATA_ROOT}.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

def decode_hdf_string(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def modacor_ready_path(referenced_path, *, relative_to):
    referenced_path = Path(decode_hdf_string(referenced_path))
    if not referenced_path.is_absolute():
        referenced_path = relative_to / referenced_path
    if not referenced_path.stem.endswith("_modacor"):
        referenced_path = referenced_path.with_name(
            f"{referenced_path.stem}_modacor{referenced_path.suffix}"
        )
    return referenced_path.resolve()


measurement_pairs = []
for candidate in sorted(DATA_ROOT.glob(SAMPLE_GLOB)):
    with h5py.File(candidate, "r") as h5:
        batch = int(h5["/entry1/experiment/batchnum"][()])
        if not SAMPLE_BATCH_START <= batch <= SAMPLE_BATCH_END:
            continue
        configuration = int(h5["/entry1/instrument/configuration"][()])
        background_file = modacor_ready_path(
            h5[BACKGROUND_REFERENCE_PATH][()], relative_to=candidate.parent
        )
        dispersant_reference = decode_hdf_string(h5[DISPERSANT_REFERENCE_PATH][()]).strip()
        dispersant_file = (
            modacor_ready_path(dispersant_reference, relative_to=candidate.parent)
            if dispersant_reference else None
        )
        displaced_dispersant_factor = float(h5[DISPLACED_DISPERSANT_FACTOR_PATH][()].mean())
        if EMBEDDED_MASK_PATH not in h5:
            raise KeyError(f"Embedded mask missing from {candidate.name}: {EMBEDDED_MASK_PATH}")

    if not background_file.is_file():
        raise FileNotFoundError(f"Background referenced by {candidate.name} is missing: {background_file}")
    with h5py.File(background_file, "r") as background_h5:
        background_configuration = int(background_h5["/entry1/instrument/configuration"][()])
    if background_configuration != configuration:
        raise ValueError(
            f"Configuration mismatch for {candidate.name}: sample {configuration}, "
            f"background {background_configuration}"
        )
    use_dispersant_pipeline = dispersant_file is not None and 0.0 < displaced_dispersant_factor < 1.0
    if use_dispersant_pipeline and not dispersant_file.is_file():
        raise FileNotFoundError(
            f"Dispersant background referenced by {candidate.name} is missing: {dispersant_file}"
        )
    measurement_pairs.append(
        {
            "sample": candidate.resolve(),
            "background": background_file,
            "dispersant": dispersant_file,
            "displaced_dispersant_factor": displaced_dispersant_factor,
            "use_dispersant_pipeline": use_dispersant_pipeline,
            "batch": batch,
            "configuration": configuration,
        }
    )

if not measurement_pairs:
    raise FileNotFoundError(f"No {SAMPLE_GLOB} files found below {DATA_ROOT}")

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Found {len(measurement_pairs)} sample/background pair(s) for batches {SAMPLE_BATCH_START}–{SAMPLE_BATCH_END}.")
for pair in measurement_pairs:
    route = "displaced-dispersant pipeline" if pair["use_dispersant_pipeline"] else "standard solids pipeline"
    print(f"  config {pair['configuration']}: {pair['sample'].name} -> {pair['background'].name} ({route})")


## Build MoDaCor sources for a measurement pair

The current pipeline uses the source references `sample` and `background`. This helper creates those sources from one resolved pair and will also be reusable by the runtime-processing loop. It additionally registers `dispersant` when metadata select the future displaced-dispersant pipeline. The external `mask_file` metadata are deliberately ignored because the usable mask is embedded at `/entry1/instrument/mask/Mask`.


In [ ]:
from modacor.io.hdf.hdf_source import HDFSource
from modacor.io.io_sources import IoSources


def build_pair_sources(pair):
    sources = IoSources()
    sources.register_source(
        HDFSource(source_reference="sample", resource_location=pair["sample"])
    )
    sources.register_source(
        HDFSource(source_reference="background", resource_location=pair["background"])
    )
    if pair["use_dispersant_pipeline"]:
        sources.register_source(
            HDFSource(source_reference="dispersant", resource_location=pair["dispersant"])
        )
    return sources


preview_pair = measurement_pairs[0]
preview_sources = build_pair_sources(preview_pair)
sample_shape = preview_sources.get_data_shape("sample::entry1/instrument/detector00/data")
background_shape = preview_sources.get_data_shape("background::entry1/instrument/detector00/data")
if sample_shape != background_shape:
    raise ValueError(f"Sample/background detector-shape mismatch: {sample_shape} versus {background_shape}")
print(f"Registered sample and background sources for configuration {preview_pair['configuration']}.")
print(f"Detector data shape: {sample_shape}")


## Preliminary pipeline graph

Loading and preparing the pipeline validates its module names, configuration models, and dependency graph before any data are processed.


In [ ]:
from IPython.display import Markdown, display
from modacor.runner.pipeline import Pipeline

pipeline = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH)
pipeline.prepare()
mermaid_source = pipeline.to_mermaid(direction="TD")
display(Markdown(f"```mermaid\n{mermaid_source}\n```"))
print(f"Prepared {len(pipeline.graph)} pipeline steps from {PIPELINE_PATH.name}.")


## Run one pair through the current pipeline

This local smoke test uses the first selected configuration and deliberately has no output sink. It verifies the complete correction chain before runtime sessions and batch output are introduced.


In [ ]:
from modacor.dataclasses.processing_data import ProcessingData
from modacor.io.buffer.runtime_buffer_store import RuntimeBufferStore
from modacor.io.io_sinks import IoSinks
from modacor.io.visualization.plotly_json_sink import PlotlyJSONSink

processing_data = ProcessingData()
smoke_plot_store = RuntimeBufferStore()
smoke_sinks = IoSinks()
smoke_sinks.register_sink(
    PlotlyJSONSink(sink_reference="plots", session_id="mouse-smoke", buffer_store=smoke_plot_store)
)
pipeline._reinitialize()
pipeline.prepare()
completed_steps = []

while pipeline.is_active():
    for step in pipeline.get_ready():
        step.io_sources = preview_sources
        step.io_sinks = smoke_sinks
        step(processing_data)
        completed_steps.append(step.step_id)
        pipeline.done(step)

corrected_signal = processing_data["sample"]["signal"].signal
q = processing_data["sample"]["Q"].signal
print(f"Completed {len(completed_steps)} steps for configuration {preview_pair['configuration']}.")
print(f"Corrected signal shape: {corrected_signal.shape}; Q shape: {q.shape}")


## Runtime server

The server retains one session per instrument configuration. The notebook starts a local server only when one is not already listening, and only stops a process that it started itself.


In [ ]:
import atexit
import subprocess
import time

import requests

BASE_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"
SERVER_PROCESS = None


def api_request(method, path, *, payload=None, expected=(200, 201, 202, 204), timeout=120):
    response = requests.request(method, BASE_URL + path, json=payload, timeout=timeout)
    if response.status_code not in expected:
        raise RuntimeError(f"{method} {path} failed ({response.status_code}): {response.text}")
    return response.json() if response.content else None


def server_ready(timeout=1):
    try:
        return requests.get(BASE_URL + "/v1/readiness", timeout=timeout).status_code == 200
    except requests.RequestException:
        return False


def start_server(timeout=45):
    global SERVER_PROCESS
    if server_ready():
        print(f"Using existing server at {BASE_URL}.")
        return
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    log = open(OUTPUT_DIR / "modacor_server.log", "a", buffering=1)
    SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "-m", "modacor.cli", "serve", "--host", SERVER_HOST, "--port", str(SERVER_PORT)],
        stdout=log, stderr=subprocess.STDOUT, text=True,
    )
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if server_ready():
            print(f"Started server at {BASE_URL}.")
            return
        if SERVER_PROCESS.poll() is not None:
            raise RuntimeError(f"Server exited early; see {OUTPUT_DIR / 'modacor_server.log'}")
        time.sleep(0.5)
    raise TimeoutError(f"Server did not become ready at {BASE_URL}")


def stop_server():
    global SERVER_PROCESS
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        SERVER_PROCESS.wait(timeout=10)
        print("Stopped notebook-owned server.")
    SERVER_PROCESS = None


atexit.register(stop_server)
start_server()


## Process the selected batches

A fresh session is created for each configuration. Multiple selected sample batches with the same configuration reuse that session. Displaced-dispersant inputs are detected but intentionally rejected until their dedicated pipeline is available.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
session_ids = {}
session_seeded = {}
batch_results = []

for pair in measurement_pairs:
    if pair["use_dispersant_pipeline"]:
        raise NotImplementedError(
            f"{pair['sample'].name} requires the future displaced-dispersant pipeline"
        )

    configuration = pair["configuration"]
    session_id = session_ids.setdefault(configuration, f"mouse-{configuration}")
    if configuration not in session_seeded:
        api_request("DELETE", f"/v1/sessions/{session_id}", expected=(204, 404), timeout=30)
        api_request(
            "POST", "/v1/sessions",
            payload={
                "session_id": session_id,
                "name": f"MOUSE configuration {configuration}",
                "source_profile": "mouse",
                "pipeline": {"yaml_path": str(PIPELINE_PATH)},
                "trace": {
                    "enabled": TRACE_ENABLED,
                    "watch": {"sample": ["signal"], "background": ["signal"]},
                    "record_only_on_change": True,
                    "snapshot_processing_data": False,
                    "snapshot_step_ids": [],
                },
                "auto_full_reset_on_partial_error": True,
            },
        )
        api_request(
            "PUT", f"/v1/sessions/{session_id}/sinks",
            payload={"sinks": [
                {"ref": "plots", "type": "plotly_json", "location": "buffer://session"}
            ]},
        )
        session_seeded[configuration] = False

    api_request(
        "PUT", f"/v1/sessions/{session_id}/sources",
        payload={"sources": [
            {"ref": "sample", "type": "hdf", "location": str(pair["sample"])},
            {"ref": "background", "type": "hdf", "location": str(pair["background"])},
        ]},
    )

    mode = "auto" if session_seeded[configuration] else "full"
    output_path = OUTPUT_DIR / f"{pair['sample'].stem}_result.h5"
    process_payload = {
        "mode": mode,
        "run_name": pair["sample"].stem,
        "rollback_snapshot": False,
        "write_hdf": {"path": str(output_path), "data_paths": OUTPUT_DATA_PATHS},
    }
    if mode == "auto":
        process_payload["changed_sources"] = ["sample", "background"]
    result = api_request("POST", f"/v1/sessions/{session_id}/process", payload=process_payload)
    session_seeded[configuration] = True
    batch_results.append({"pair": pair, "output": output_path, "result": result})
    print(f"config {configuration}: {result['status']} -> {output_path.name}")

print(f"Completed {len(batch_results)} measurement pair(s) in {len(session_ids)} session(s).")


## Live plots

Each configuration session publishes its latest corrected 1D and 2D Plotly figures through the registered `plots` IoSink. These links remain available while the runtime server is running.


In [ ]:
plot_links = ["## Configuration plots"]
for configuration, session_id in sorted(session_ids.items()):
    plot_links.extend([
        f"### Configuration {configuration}",
        f"- [Corrected I(Q)]({BASE_URL}/v1/sessions/{session_id}/plots/plots/mouse-1d)",
        f"- [Corrected 2D detector image]({BASE_URL}/v1/sessions/{session_id}/plots/plots/mouse-2d)",
    ])
display(Markdown("\n".join(plot_links)))


## Next milestone

The standard plate-sample pipeline is now complete enough for comparison and refinement. A separate capillary self-absorption model and displaced-dispersant pipeline remain future work.


## Optional: stop the runtime server

Run this cleanup cell when the runtime sessions and live plots are no longer needed. The plot links above will stop working until the server is started again. The helper only stops a server process started by this notebook; it leaves an independently started server alone.


In [ ]:
stop_server()
